# AIMS Africa Multilingual Tokenizer Challenge — Baseline `BPE 10K`

**Objectif de ce notebook :** construire la **première baseline** (uniquement une baseline, aucune optimisation).

**Pipeline exact :**

```text
Dataset officiel  →  NFC  →  Whitespace  →  BPE  →  10 000 tokens max  →  Évaluation
```

**Règles impératives appliquées dans ce notebook**
- Dataset **officiel uniquement** : `Similoluwa/african-multilingual-tokenizer-challenge`, révision `v1.0.0` (Hugging Face).
- Entraînement **uniquement** sur `dataset["train"]` ; évaluation **uniquement** sur `dataset["validation"]`.
- Le dataset est déjà normalisé **NFC** et les diacritiques linguistiques sont conservés :
  **aucune** conversion ASCII, **aucun** `StripAccents`, **aucun** NFD / NFKD + suppression d'accents, **aucun** `lowercase`.
- Aucun score n'est inventé : toutes les valeurs affichées et écrites dans `reports/` sont calculées
  à partir du dataset réel au moment de l'exécution.

> Exécution : Google Colab, de haut en bas (menu `Runtime` → `Run all`).

## Installation (Colab)

Rien d'autre n'est nécessaire : le dataset est chargé directement depuis Hugging Face `datasets` (aucun téléchargement manuel).

In [ ]:
!pip install -q datasets tokenizers pandas numpy

## 1. Configuration globale (baseline figée)

Ne **rien modifier** pour cette baseline : `vocab_size = 10 000`, `min_frequency = 2`,
spécial `[UNK]`, normaliseur `NFC`, pré-tokeniseur `Whitespace`, modèle `BPE`.

In [ ]:
# =============================================================================
# 1. Configuration globale (baseline BPE 10K — ne pas modifier)
# =============================================================================
import json
import math
import os
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

# --- Dataset officiel du challenge ----------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# --- Configuration tokenizer (baseline figée) -----------------------------
MODEL_TYPE = "BPE"
UNK_TOKEN = "[UNK]"
SPECIAL_TOKENS = [UNK_TOKEN]
VOCAB_SIZE = 10_000
MIN_FREQUENCY = 2
NORMALIZER = "NFC"
PRE_TOKENIZER = "Whitespace"

# --- Splits ----------------------------------------------------------------
TRAIN_SPLIT = "train"
EVAL_SPLIT = "validation"

# --- Langues ---------------------------------------------------------------
LANG_ORDER = ["en", "fr", "ha", "sw", "yo", "am"]
LANGUAGE_NAMES = {
    "en": "English", "fr": "French", "ha": "Hausa",
    "sw": "Swahili", "yo": "Yoruba", "am": "Amharic",
}
TARGET_LANGUAGES = ["ha", "sw", "yo", "am"]  # langues cibles pour la moyenne "Target average"

# --- Comptes attendus (fiche officielle du dataset, révision v1.0.0) -------
EXPECTED = {
    "train":      {"en": 40000, "fr": 40000, "ha": 40000, "sw": 40000, "yo": 40000, "am": 40000, "total": 240000},
    "validation": {"en":  4000, "fr":  4000, "ha":  4000, "sw":  4000, "yo":  4000, "am":  4000, "total":  24000},
}

# --- Répertoires de sortie --------------------------------------------------
# En Colab le répertoire courant est /content : les artefacts sont écrits dans
# /content/models/... et /content/reports/... (à télécharger ensuite).
OUTPUT_ROOT = Path.cwd()
MODEL_DIR = OUTPUT_ROOT / "models" / "baseline_bpe_10k"
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_FILE = MODEL_DIR / "tokenizer.json"
REPORT_JSON = REPORT_DIR / "baseline_bpe_10k.json"
REPORT_MD = REPORT_DIR / "baseline_bpe_10k.md"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Répertoire courant      :", OUTPUT_ROOT)
print("Fichier tokenizer       :", MODEL_FILE)
print("Rapports                :", REPORT_JSON, "|", REPORT_MD)

# --- Définition de "mot" (identique pour toutes les langues) ---------------
def split_words(text: str):
    """Un mot = toute suite maximale de caractères non-espaces (text.split()).

    Cette définition est identique pour les 6 langues et alignée sur le
    pré-tokeniseur Whitespace (un mot produit toujours au moins 1 pré-token).
    Elle inclut la ponctuation attachée (ex. 'café,'), qui compte ensuite pour
    son propre token dans le flux BPE.
    """
    return text.split()

# --- Reproducibilité --------------------------------------------------------
RNG_SEED = 0
RNG = random.Random(RNG_SEED)

# --- Versions ---------------------------------------------------------------
import datasets
import numpy as np
import pandas as pd
import tokenizers

print("datasets   :", datasets.__version__)
print("tokenizers :", tokenizers.__version__)
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)

## 2. Chargement du dataset officiel

Chargement direct depuis Hugging Face `datasets`, révision **`v1.0.0`** (aucun chemin local,
aucun téléchargement manuel).

In [ ]:
# =============================================================================
# 2. Chargement du dataset officiel (révision v1.0.0)
# =============================================================================
from datasets import load_dataset

t0 = time.time()
dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(f"Dataset chargé en {time.time() - t0:.1f}s")
print(dataset)

print("\nSplits :", list(dataset.keys()))
for split_name in dataset:
    ds = dataset[split_name]
    print(f"  {split_name} : {len(ds):,} exemples | colonnes : {ds.column_names}")

print("\nAperçu du dataset :")
dataset

## 3. Inspection automatique du dataset

Vérifications automatiques : colonnes, nombre d'exemples, langues disponibles,
nombre d'exemples **par langue** et par split.

Valeurs attendues (fiche officielle, révision `v1.0.0`) :

```text
train      = 240 000 exemples   (40 000 par langue × 6)
validation =  24 000 exemples   ( 4 000 par langue × 6)
```

Si les nombres réels diffèrent, **rien n'est inventé** : les valeurs réelles sont affichées
et signalées par `MISMATCH`.

In [ ]:
# =============================================================================
# 3. Vérifications automatiques (colonnes, exemples, langues, répartition)
# =============================================================================
print("=== Colonnes par split ===")
for split_name in dataset:
    cols = dataset[split_name].column_names
    print(f"{split_name:12s} : {cols}")
    assert "language" in cols, "colonne 'language' absente"
    assert "text" in cols, "colonne 'text' absente"
print("OK : les colonnes 'language' et 'text' sont présentes.\n")

print("=== Nombre d'exemples par langue et par split ===")
count_rows = []
all_checks_ok = True
for split_name in [TRAIN_SPLIT, EVAL_SPLIT]:
    ds = dataset[split_name]
    counts = Counter(ds["language"])
    for lang in LANG_ORDER:
        real = counts.get(lang, 0)
        expected = EXPECTED[split_name][lang]
        ok = real == expected
        all_checks_ok &= ok
        count_rows.append({
            "split": split_name, "language": lang, "examples": real,
            "expected": expected, "match": "OK" if ok else "MISMATCH",
        })
        print(f"  {split_name:10s} {lang:2s} : {real:>7,} exemples"
              f" (attendu {expected:,}) -> {'OK' if ok else 'MISMATCH'}")
    real_total = len(ds)
    expected_total = EXPECTED[split_name]["total"]
    ok_total = real_total == expected_total
    all_checks_ok &= ok_total
    print(f"  {split_name:10s} total : {real_total:>7,} exemples"
          f" (attendu {expected_total:,}) -> {'OK' if ok_total else 'MISMATCH'}")
    print("  langues présentes :", sorted(counts.keys()))

print("\nRésultat global :", "OK — les comptes correspondent à la fiche officielle."
      if all_checks_ok else "ATTENTION — certains comptes diffèrent de la fiche (valeurs réelles affichées ci-dessus).")
counts_df = pd.DataFrame(count_rows)
counts_df

In [ ]:
# =============================================================================
# 3bis. Exemples représentatifs (un par langue, split train)
# =============================================================================
print("=== Exemples représentatifs (dataset['train']) ===\n")
for lang in LANG_ORDER:
    texts = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
    sample = texts[:1] + texts[len(texts) // 2: len(texts) // 2 + 1]
    for t in sample:
        print(f"[{lang} · {LANGUAGE_NAMES[lang]}] ({len(t)} chars)")
        print("   ", t)
    print()

## 4. Statistiques du dataset

Pour chaque langue et chaque split : nombre d'exemples, nombre total de mots,
nombre total de caractères, longueur moyenne en mots, longueur moyenne en
caractères et nombre de caractères Unicode distincts.

In [ ]:
# =============================================================================
# 4. Statistiques par langue et par split
# =============================================================================
char_counters = {}  # (split, lang) -> Counter des caractères (réutilisé plus bas)

def compute_language_stats(texts):
    n = len(texts)
    total_words = 0
    total_chars = 0
    chars = Counter()
    for text in texts:
        total_words += len(split_words(text))
        total_chars += len(text)
        chars.update(text)
    return {
        "examples": n,
        "words": total_words,
        "characters": total_chars,
        "avg_words": round(total_words / n, 4) if n else 0.0,
        "avg_chars": round(total_chars / n, 4) if n else 0.0,
        "distinct_chars": len(chars),
        "char_counter": chars,
    }

stat_rows = []
for split_name in [TRAIN_SPLIT, EVAL_SPLIT]:
    ds = dataset[split_name]
    lang_col = ds["language"]
    text_col = ds["text"]
    for lang in LANG_ORDER:
        texts = [t for t, l in zip(text_col, lang_col) if l == lang]
        s = compute_language_stats(texts)
        char_counters[(split_name, lang)] = s["char_counter"]
        stat_rows.append({
            "language": LANGUAGE_NAMES[lang],
            "lang": lang,
            "split": split_name,
            "examples": s["examples"],
            "words": s["words"],
            "characters": s["characters"],
            "avg_words": s["avg_words"],
            "avg_chars": s["avg_chars"],
            "distinct_unicode_chars": s["distinct_chars"],
        })

stats_df = pd.DataFrame(stat_rows)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
print("Statistiques par langue et par split :")
stats_df[["language", "split", "examples", "words", "characters",
          "avg_words", "avg_chars", "distinct_unicode_chars"]]

## 5. Vérification Unicode (6 langues)

Analyse des caractères : caractères Unicode distincts, caractères non ASCII,
caractères les plus fréquents, exemples de texte. Attention particulière sur
l'**amharique** (syllabaire éthiopien) et les **diacritiques du yoruba**
(ẹ, ọ, ṣ + tons).

Comme demandé pour cette baseline, **aucune** transformation n'est appliquée :
pas de conversion ASCII, pas de `StripAccents`, pas de NFD + suppression, pas de `lowercase`.

In [ ]:
# =============================================================================
# 5. Vérification Unicode par langue (train)
# =============================================================================
def char_label(ch):
    try:
        return unicodedata.name(ch, "?")
    except ValueError:
        return "?"

print("=== Caractères Unicode distincts (train) ===\n")
for lang in LANG_ORDER:
    counter = char_counters[("train", lang)]
    total_chars = sum(counter.values())
    non_ascii = sorted(ch for ch in counter if ord(ch) > 127)
    # Vérification NFC sur un échantillon déterministe de 2 000 textes
    texts = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang][:2000]
    not_nfc = sum(1 for t in texts if not unicodedata.is_normalized("NFC", t))
    print(f"--- {LANGUAGE_NAMES[lang]} ({lang}) ---")
    print(f"  caractères distincts      : {len(counter):,}")
    print(f"  caractères distincts non-ASCII : {len(non_ascii):,}")
    print("  caractères les plus fréquents :",
          ", ".join(f"{ch!r}({n})" for ch, n in counter.most_common(12)))
    print(f"  échantillon non-ASCII     : {''.join(non_ascii[:50])!r}")
    print(f"  textes non-NFC (sur 2000)  : {not_nfc}")
    print()

# --- Points d'attention demandés --------------------------------------------
print("=== Amharique (am) ===")
am_counter = char_counters[("train", "am")]
ethiopic = sorted(ch for ch in am_counter if 0x1200 <= ord(ch) <= 0x137F)
print(f"  caractères du syllabaire éthiopien (U+1200..U+137F) : {len(ethiopic):,}")
print("  exemples :", "".join(ethiopic[:40]))
am_texts = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == "am"]
print("  exemple de texte :")
print("   ", am_texts[0])
print()

print("=== Diacritiques du Yoruba (yo) ===")
yo_counter = char_counters[("train", "yo")]
latin_letters = []
for ch in yo_counter:
    name = unicodedata.name(ch, "")
    if name.startswith("LATIN") and ("WITH" in name):
        latin_letters.append(ch)
print("  lettres latines à diacritique distinctes :",
      ", ".join(f"{ch!r} U+{ord(ch):04X}" for ch in sorted(latin_letters)))
yo_texts = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == "yo"]
print("  exemple de texte :")
print("   ", yo_texts[0])
print()

print("=== Hausa (ha) : lettres à crochet (ɓ ɗ ƙ) ===")
ha_counter = char_counters[("train", "ha")]
hooks = sorted(ch for ch in ha_counter if unicodedata.name(ch, "").startswith("LATIN") and ("HOOK" in unicodedata.name(ch, "") or ch in "ɓɗƙ"))
print("  lettres à crochet :", "".join(hooks) or "(aucune détectée)")

## 6. Construction du tokenizer (configuration exacte de la baseline)

```python
Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = NFC()
tokenizer.pre_tokenizer = Whitespace()
BpeTrainer(vocab_size=10_000, min_frequency=2, special_tokens=["[UNK]"])
```

In [ ]:
# =============================================================================
# 6. Tokenizer + Trainer (baseline figée)
# =============================================================================
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
tokenizer.normalizer = NFC()
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
)

print("Tokenizer configuré :")
print("  modèle          :", MODEL_TYPE, f"(unk_token={UNK_TOKEN!r})")
print("  normaliseur     :", NORMALIZER)
print("  pré-tokeniseur  :", PRE_TOKENIZER)
print("  vocab_size      :", VOCAB_SIZE)
print("  min_frequency   :", MIN_FREQUENCY)
print("  special_tokens  :", SPECIAL_TOKENS)

## 7. Entraînement du tokenizer

Entraînement **uniquement sur `dataset["train"]`** (jamais sur `dataset["validation"]`),
sur l'ensemble multilingue complet, via `train_from_iterator()`.

In [ ]:
# =============================================================================
# 7. Entraînement BPE sur dataset["train"] uniquement
# =============================================================================
def corpus_iterator(dataset_split, log_every=40_000):
    """Itère sur les textes du split (yield text) avec un suivi de progression."""
    n = len(dataset_split)
    for i, text in enumerate(dataset_split["text"]):
        yield text
        if (i + 1) % log_every == 0:
            print(f"  ... {i + 1:,}/{n:,} textes fournis à l'entraîneur")

print(f"Entraînement sur dataset['{TRAIN_SPLIT}'] "
      f"({len(dataset[TRAIN_SPLIT]):,} textes multilingues)...")
t0 = time.time()
tokenizer.train_from_iterator(corpus_iterator(dataset[TRAIN_SPLIT]), trainer=trainer)
dt = time.time() - t0
print(f"Entraînement terminé en {dt:.1f}s ({dt / 60:.2f} min)")

## 8. Vocabulaire réel

Affiche la taille de vocabulaire demandée et la taille réelle. Le vocabulaire ne
doit **jamais dépasser 10 000**.

In [ ]:
# =============================================================================
# 8. Vocabulaire : demandé vs réel (<= 10 000)
# =============================================================================
actual_vocab_size = tokenizer.get_vocab_size()
print("Requested vocab size:", VOCAB_SIZE)
print("Actual vocab size:   ", actual_vocab_size)

if actual_vocab_size > VOCAB_SIZE:
    print("ERREUR : le vocabulaire dépasse la limite demandée.")
else:
    print(f"OK : le vocabulaire ({actual_vocab_size}) <= {VOCAB_SIZE}.")
    if actual_vocab_size < VOCAB_SIZE:
        print("Note : l'entraîneur s'est arrêté avant d'atteindre 10 000 (pas assez de fusions"
              " distinctes avec min_frequency=2) : la valeur réelle ci-dessus fait foi.")

vocab = tokenizer.get_vocab()
unk_id = tokenizer.token_to_id(UNK_TOKEN)
print(f"\n{UNK_TOKEN} -> id {unk_id}")
print("\nPremières entrées du vocabulaire :")
for token, tid in sorted(vocab.items(), key=lambda kv: kv[1])[:15]:
    print(f"  {tid:5d}  {token!r}")

## 9. Sauvegarde et rechargement

Le tokenizer est sauvegardé dans `models/baseline_bpe_10k/tokenizer.json`, puis
rechargé avec `Tokenizer.from_file(...)`. Les deux objets doivent produire les
mêmes tokens sur plusieurs exemples.

In [ ]:
# =============================================================================
# 9. Sauvegarde + rechargement + vérification d'égalité des tokens
# =============================================================================
# Tokenizer.save() écrit directement models/baseline_bpe_10k/tokenizer.json.
# Repli sur save_pretrained() (même fichier tokenizer.json) si l'API venait à
# changer selon la version de `tokenizers` installée.
if hasattr(tokenizer, "save"):
    tokenizer.save(str(MODEL_FILE))
    print("Sauvegardé (Tokenizer.save) :", MODEL_FILE)
else:
    tokenizer.save_pretrained(str(MODEL_DIR))
    print("Sauvegardé (Tokenizer.save_pretrained) :", MODEL_FILE)

reloaded = Tokenizer.from_file(str(MODEL_FILE))
print("Rechargé depuis :", str(MODEL_FILE))

# Cohérence tokenizer <-> reloaded sur plusieurs exemples (6 langues × 2 splits)
consistency_examples = []
for split_name in [TRAIN_SPLIT, EVAL_SPLIT]:
    ds = dataset[split_name]
    for lang in LANG_ORDER:
        texts = [t for t, l in zip(ds["text"], ds["language"]) if l == lang]
        consistency_examples.extend(texts[i] for i in [0, len(texts) // 2])

n_diff = 0
for text in consistency_examples:
    tok_a = tokenizer.encode(text, add_special_tokens=False).tokens
    tok_b = reloaded.encode(text, add_special_tokens=False).tokens
    if tok_a != tok_b:
        n_diff += 1
        print("DIFFÉRENCE détectée sur :", repr(text[:80]))

if n_diff == 0:
    print(f"OK : tokenizer et reloaded produisent des tokens identiques sur "
          f"{len(consistency_examples)} exemples.")
else:
    print(f"ATTENTION : {n_diff} exemples divergent.")

## 10. Évaluation sur le split `validation`

**Le `train` sert uniquement à entraîner le tokenizer.** L'évaluation de la baseline
se fait exclusivement sur `dataset["validation"]` (4 000 textes par langue).

Méthode — clairement définie et **identique pour les 6 langues** :

- **Words** : nombre de mots, où un mot = toute suite maximale de caractères non-espaces
  (`text.split()`, cf. définition `split_words` plus haut). Chaque mot produit au moins
  un pré-token avec le pré-tokeniseur `Whitespace`.
- **Tokens** : nombre total de tokens produits par `tokenizer.encode_batch(..., add_special_tokens=False)`
  (aucun token spécial ajouté).
- **Fertility** : `fertility = total_tokens / total_words`.
- **UNK** : nombre total de `[UNK]` produits.
- **UNK rate** : `unk_rate = unk_count / total_words`.
- **Score** : `score = fertility + 100 * unk_rate`.

Fonction d'évaluation générique : `evaluate_language(tokenizer, texts)`.

In [ ]:
# =============================================================================
# 10. Fonction d'évaluation + calcul des métriques sur validation
# =============================================================================
def evaluate_language(tokenizer, texts, batch_size=2048):
    """Métriques d'une langue : Words, Tokens, Fertility, UNK, UNK rate, Score.

    Texts : liste des textes de validation d'UNE langue.
    """
    total_words = 0
    total_tokens = 0
    unk_count = 0
    token_freq = Counter()

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        total_words += sum(len(split_words(t)) for t in batch)
        encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
        for enc in encodings:
            ids = enc.ids
            total_tokens += len(ids)
            n_unk = ids.count(unk_id)
            unk_count += n_unk
            token_freq.update(enc.tokens)

    fertility = total_tokens / total_words if total_words else float("nan")
    unk_rate = unk_count / total_words if total_words else float("nan")
    return {
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "unk": unk_count,
        "unk_rate": unk_rate,
        "score": fertility + 100.0 * unk_rate,
        "token_freq": token_freq,
    }

# --- Textes de validation par langue (une seule passe de filtrage) ----------
val_texts = {}
print("Préparation des textes de validation par langue :")
for lang in LANG_ORDER:
    texts = [t for t, l in zip(dataset[EVAL_SPLIT]["text"], dataset[EVAL_SPLIT]["language"]) if l == lang]
    val_texts[lang] = texts
    n = len(texts)
    exp = EXPECTED[EVAL_SPLIT][lang]
    status = "OK" if n == exp else "MISMATCH"
    print(f"  {lang} ({LANGUAGE_NAMES[lang]}): {n:,} textes (attendu {exp:,}) -> {status}")

# --- Évaluation par langue --------------------------------------------------
print("\nÉvaluation sur dataset['validation'] ...")
results = {}
for lang in LANG_ORDER:
    t0 = time.time()
    r = evaluate_language(tokenizer, val_texts[lang])
    r["language"] = LANGUAGE_NAMES[lang]
    r["lang"] = lang
    results[lang] = r
    print(f"  {lang} ({LANGUAGE_NAMES[lang]}) évalué en {time.time() - t0:.1f}s")

# --- Tableau final (Section 9) ----------------------------------------------
table_rows = []
for lang in LANG_ORDER:
    r = results[lang]
    table_rows.append({
        "Language": r["language"],
        "Words": f"{r['words']:,}",
        "Tokens": f"{r['tokens']:,}",
        "Fertility": round(r["fertility"], 4),
        "UNK": r["unk"],
        "UNK Rate": round(r["unk_rate"], 6),
        "Score": round(r["score"], 4),
    })
results_table = pd.DataFrame(table_rows)
print("\n=== Tableau final : Language | Words | Tokens | Fertility | UNK | UNK Rate | Score ===")
results_table

## 11. Score cible — moyenne sur les langues du challenge

La moyenne cible (`Target average`) est calculée **séparément** sur les 4 langues
africaines du challenge :

```text
Hausa, Swahili, Yoruba, Amharic
```

`target_average = moyenne des scores des langues cibles` (chaque langue compte à
parts égales).

In [ ]:
# =============================================================================
# 11. Target average (ha, sw, yo, am)
# =============================================================================
target_avg_score = sum(results[l]["score"] for l in TARGET_LANGUAGES) / len(TARGET_LANGUAGES)
target_avg_fertility = sum(results[l]["fertility"] for l in TARGET_LANGUAGES) / len(TARGET_LANGUAGES)
target_avg_unk_rate = sum(results[l]["unk_rate"] for l in TARGET_LANGUAGES) / len(TARGET_LANGUAGES)

print("Hausa  :", round(results["ha"]["score"], 4))
print("Swahili:", round(results["sw"]["score"], 4))
print("Yoruba :", round(results["yo"]["score"], 4))
print("Amharic:", round(results["am"]["score"], 4))
print("----------------")
print("Target average (score)     :", round(target_avg_score, 4))
print("Target average (fertility) :", round(target_avg_fertility, 4))
print("Target average (UNK rate)  :", round(target_avg_unk_rate, 6))

## 12. English / French — guardrails officiels (règle vérifiée)

Le challenge définit un guardrail **relatif** (et non un seuil absolu). Constantes officielles
extraites de `competition/constants.py` du dépôt
[`airf-multilingual-tokenizer-challenge`](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge) :

```python
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15

budget = 1.15 * moyenne(fertility BRUTE de ha, sw, yo, am)
# échec si fertility["en"] > budget OU fertility["fr"] > budget
```

Un dépassement rend la soumission **invalide** : l'évaluateur officiel lève
`context-language guardrail exceeded` (`competition/evaluation.py`).

> ⚠️ Le budget dépend de **votre propre** moyenne sur les langues notées : améliorer fortement
> ha/sw/yo/am **abaisse** le budget et peut donc casser le guardrail. À surveiller à chaque
> expérience (c'est géré automatiquement dans `02_optimization_sweep.ipynb`).

In [ ]:
# =============================================================================
# 12. English / French + guardrail OFFICIEL (règle relative, vérifiée)
# =============================================================================
# Règle officielle (competition/constants.py + competition/metrics.py) :
#   budget = 1.15 x moyenne(fertility brute des langues notees : ha, sw, yo, am)
#   echec si fertility[en] > budget ou fertility[fr] > budget
CONTEXT_LANGUAGES = ["en", "fr"]
CONTEXT_FERTILITY_RATIO = 1.15


def official_guardrail(fertility_by_lang):
    """Retourne (moyenne brute des langues notées, budget, langues en dépassement)."""
    raw = sum(fertility_by_lang[l] for l in TARGET_LANGUAGES) / len(TARGET_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility_by_lang.get(l, 0.0) > budget]
    return raw, budget, breaches


en = results["en"]
fr = results["fr"]

for label, r in [("English", en), ("French", fr)]:
    print(f"{label} : words={r['words']:,} | tokens={r['tokens']:,} | "
          f"fertility={r['fertility']:.4f} | unk={r['unk']} | "
          f"unk_rate={r['unk_rate']:.6f} | score={r['score']:.4f}")

raw_scored, guardrail_budget, guardrail_breaches = official_guardrail(
    {l: results[l]["fertility"] for l in LANG_ORDER})

print()
print(f"Moyenne fertility brute (ha, sw, yo, am) : {raw_scored:.6f}")
print(f"Budget guardrail = {raw_scored:.6f} x {CONTEXT_FERTILITY_RATIO} = {guardrail_budget:.6f}")
for code in CONTEXT_LANGUAGES:
    fertility = results[code]["fertility"]
    verdict = "PASS" if fertility <= guardrail_budget else "FAIL"
    print(f"  {LANGUAGE_NAMES[code]} : fertility {fertility:.6f} vs budget {guardrail_budget:.6f} "
          f"-> {verdict} (marge {guardrail_budget - fertility:+.4f})")
print()
print("English guardrail:", "FAIL" if "en" in guardrail_breaches else "PASS")
print("French guardrail :", "FAIL" if "fr" in guardrail_breaches else "PASS")
print("Dépassements :", guardrail_breaches or "aucun")

## 13. Exemples de tokenisation (≥ 10 par langue)

Pour chacune des six langues : `Original` / `Tokens` / `Token IDs`.
La sélection (déterministe, graine fixée) favorise les textes contenant :
caractères accentués, diacritiques, ponctuation, mots longs, mots rares et texte amharique.

In [ ]:
# =============================================================================
# 13. Exemples de tokenisation — >= 10 exemples par langue
# =============================================================================
def select_demo_indices(texts, rng, k=10):
    """Sélection déterministe de k indices parmi texts (critères : non-ASCII,
    ponctuation, mots longs, longueur ; puis complément aléatoire déterministe)."""
    def interest(t):
        score = 0
        if any(ord(c) > 127 for c in t):
            score += 100                      # script / accents / diacritiques
        if any(c in ".,;:!?()[]{}\"'«»“”‘’—…።፣،؟" for c in t):
            score += 10                       # ponctuation
        if any(len(w) >= 12 for w in split_words(t)):
            score += 10                       # mot long
        if len(t) >= 150:
            score += 5
        return score

    ranked = sorted(range(len(texts)), key=lambda i: (-interest(texts[i]), i))
    chosen = ranked[: k // 2]
    pool = [i for i in range(len(texts)) if i not in set(chosen)]
    chosen = chosen + rng.sample(pool, min(len(pool), k - len(chosen)))
    return chosen[:k]

for lang in LANG_ORDER:
    print("=" * 90)
    print(f"{LANGUAGE_NAMES[lang]} ({lang}) — {len(val_texts[lang]):,} textes de validation")
    print("=" * 90)
    for idx in select_demo_indices(val_texts[lang], RNG, k=10):
        text = val_texts[lang][idx]
        enc = tokenizer.encode(text, add_special_tokens=False)
        print("Original:")
        print("  ", text)
        print("Tokens:")
        print("  ", enc.tokens)
        print("Token IDs:")
        print("  ", enc.ids)
        print()

## 14. Analyse des `[UNK]`

Si `[UNK]` apparaît, cette section identifie les exemples responsables :

```text
language | text | problematic_token | unk_count
```

et explique la cause racine : un `[UNK]` apparaît quand un caractère présent dans
le texte n'appartient pas au vocabulaire BPE. Avec `min_frequency=2`, un caractère
doit apparaître au moins 2 fois dans tout le train pour entrer dans l'alphabet de
base du BPE.

In [ ]:
# =============================================================================
# 14. Analyse des [UNK] sur validation (si présents)
# =============================================================================
def find_unk_rows(tokenizer, texts, lang, vocab_set, train_char_counter, max_rows=5, batch_size=2048):
    """Retourne jusqu'à max_rows lignes contenant au moins un [UNK]."""
    rows = []
    word_re = re.compile(r"\S+")
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
        for text, enc in zip(batch, encodings):
            if any(tok == UNK_TOKEN for tok in enc.tokens):
                spans = list(word_re.finditer(text))
                problem_words = []
                for i, tok in enumerate(enc.tokens):
                    if tok == UNK_TOKEN:
                        s, _e = enc.offsets[i]
                        for m in spans:
                            if m.start() <= s < m.end():
                                problem_words.append(text[m.start():m.end()])
                                break
                        else:
                            problem_words.append(text[max(0, s - 8):s + 8])
                unk_in_text = sum(1 for tok in enc.tokens if tok == UNK_TOKEN)
                # Caractères absents du vocabulaire (cause racine)
                absent_chars = sorted({ch for w in problem_words for ch in w} - vocab_set)
                rows.append({
                    "language": lang,
                    "text": text,
                    "problematic_token": " | ".join(dict.fromkeys(problem_words)),
                    "unk_count": unk_in_text,
                    "absent_chars": "".join(absent_chars),
                })
                if len(rows) >= max_rows:
                    return rows
    return rows

vocab_set = set(tokenizer.get_vocab().keys())
unk_rows = []
for lang in LANG_ORDER:
    r = results[lang]
    if r["unk"] > 0:
        rows = find_unk_rows(tokenizer, val_texts[lang], LANGUAGE_NAMES[lang], vocab_set,
                             char_counters[("train", lang)], max_rows=5)
        print(f"[{LANGUAGE_NAMES[lang]} ({lang})] {r['unk']} [UNK] sur {r['words']:,} mots "
              f"(unk_rate={r['unk_rate']:.6f})")
        for row in rows:
            print("  -", repr(row["text"][:110]))
            print(f"    problematic_token : {row['problematic_token']!r} | unk_count={row['unk_count']} | "
                  f"chars absents du vocab : {row['absent_chars']!r}")
        unk_rows.extend(rows)
    else:
        print(f"[{LANGUAGE_NAMES[lang]} ({lang})] aucun [UNK] sur le split validation.")

unk_analysis_df = pd.DataFrame([
    {k: row[k] for k in ["language", "text", "problematic_token", "unk_count"]}
    for row in unk_rows
])
if len(unk_analysis_df):
    print("\nTableau : language | text | problematic_token | unk_count")
    unk_analysis_df

print("\nExplication : si [UNK] apparaît, c'est qu'un caractère du texte est absent du vocabulaire"
      " (alphabet de base : caractères vus >= min_frequency=2 fois sur l'ensemble du train)."
      " Les caractères listés ci-dessus dans 'absent_chars' sont les responsables directs.")

## 15. Statistiques du vocabulaire

Vocabulaire total, tokens spéciaux, tokens de longueur 1/2/3, longueur moyenne et
maximale (en caractères Unicode). Les fréquences des tokens ne sont pas stockées
dans `tokenizer.json` : les "tokens les plus fréquents" sont donc estimés sur le
split de validation (24 000 textes tokenisés pendant l'évaluation), ce qui est
signalé explicitement.

In [ ]:
# =============================================================================
# 15. Statistiques du vocabulaire
# =============================================================================
vocab = tokenizer.get_vocab()
all_tokens = list(vocab.keys())

special_in_vocab = sorted(tok for tok in all_tokens if tok in SPECIAL_TOKENS)
regular_tokens = [tok for tok in all_tokens if tok not in SPECIAL_TOKENS]

len_counts = Counter(len(tok) for tok in regular_tokens)
mean_len = sum(len(tok) for tok in regular_tokens) / len(regular_tokens) if regular_tokens else 0.0
max_len = max((len(tok) for tok in regular_tokens), default=0)
longest = sorted((tok for tok in regular_tokens if len(tok) == max_len))[:10]

print(f"Vocabulaire total            : {len(all_tokens):,}")
print(f"Tokens spéciaux              : {len(special_in_vocab)} -> {special_in_vocab}")
print(f"Tokens réguliers             : {len(regular_tokens):,}")
print(f"Tokens de longueur 1         : {len_counts.get(1, 0):,}")
print(f"Tokens de longueur 2         : {len_counts.get(2, 0):,}")
print(f"Tokens de longueur 3         : {len_counts.get(3, 0):,}")
print(f"Longueur moyenne             : {mean_len:.3f} caractères")
print(f"Longueur maximale            : {max_len} caractères")
print(f"Exemples de tokens les plus longs : {longest}")

# Tokens les plus fréquents : estimation sur le split validation (voir note)
merged_freq = Counter()
for lang in LANG_ORDER:
    merged_freq.update(results[lang]["token_freq"])
print("\nTokens les plus fréquents — estimation sur dataset['validation'] (24 000 textes) :")
for tok, cnt in merged_freq.most_common(25):
    print(f"  {cnt:>8,}  {tok!r}")

vocab_stats = {
    "total_vocab": len(all_tokens),
    "special_tokens": special_in_vocab,
    "len_1": len_counts.get(1, 0),
    "len_2": len_counts.get(2, 0),
    "len_3": len_counts.get(3, 0),
    "mean_length": round(mean_len, 4),
    "max_length": max_len,
    "most_frequent_validation_estimate": merged_freq.most_common(25),
}

## 16. Rapports générés

Écrit deux rapports :
- `reports/baseline_bpe_10k.json`
- `reports/baseline_bpe_10k.md`

Contenu : dataset + révision, statistiques, configuration du tokenizer, vocabulaire
réel, résultats par langue (fertility, UNK rate, score), target average, English,
French, guardrails, exemples de tokenisation et analyse des UNK.

In [ ]:
# =============================================================================
# 16. Génération des rapports (JSON + Markdown)
# =============================================================================
per_language_report = {}
for lang in LANG_ORDER:
    r = results[lang]
    per_language_report[lang] = {
        "language": r["language"],
        "words": r["words"],
        "tokens": r["tokens"],
        "fertility": round(r["fertility"], 6),
        "unk": r["unk"],
        "unk_rate": round(r["unk_rate"], 8),
        "score": round(r["score"], 6),
    }

report = {
    "experiment": "baseline_bpe_10k",
    "note": "Aucun score n'est fabriqué : valeurs calculées au moment de l'exécution du notebook.",
    "status": "computed_on_official_dataset" if len(dataset["train"]) == EXPECTED["train"]["total"] else "computed_on_official_dataset_counts_mismatch",
    "1_dataset": {
        "name": DATASET_NAME,
        "url": "https://huggingface.co/datasets/Similoluwa/african-multilingual-tokenizer-challenge",
        "revision": DATASET_REVISION,
        "source": "Hugging Face datasets (load_dataset)",
        "columns": dataset["train"].column_names,
        "expected_counts": EXPECTED,
        "actual_counts_per_split": {s: dict(Counter(dataset[s]["language"])) for s in dataset},
    },
    "2_statistics": stats_df.drop(columns=["lang"]).to_dict(orient="records"),
    "3_tokenizer_config": {
        "model": MODEL_TYPE,
        "unk_token": UNK_TOKEN,
        "normalizer": NORMALIZER,
        "pre_tokenizer": PRE_TOKENIZER,
        "trainer": {
            "vocab_size": VOCAB_SIZE,
            "min_frequency": MIN_FREQUENCY,
            "special_tokens": SPECIAL_TOKENS,
        },
        "training_split": TRAIN_SPLIT,
        "evaluation_split": EVAL_SPLIT,
        "word_definition": "word = maximal run of non-whitespace characters (text.split())",
    },
    "4_vocabulary": {
        "requested_vocab_size": VOCAB_SIZE,
        "actual_vocab_size": actual_vocab_size,
        "within_limit": actual_vocab_size <= VOCAB_SIZE,
        "special_tokens": special_in_vocab,
        "len1_tokens": vocab_stats["len_1"],
        "len2_tokens": vocab_stats["len_2"],
        "len3_tokens": vocab_stats["len_3"],
        "mean_token_length": vocab_stats["mean_length"],
        "max_token_length": vocab_stats["max_length"],
        "most_frequent_tokens": {"note": "estimated on validation split",
                                 "top25": [[tok, cnt] for tok, cnt in vocab_stats["most_frequent_validation_estimate"]]},
    },
    "5_results_per_language": per_language_report,
    "6_fertility": {lang: per_language_report[lang]["fertility"] for lang in LANG_ORDER},
    "7_unk_rate": {lang: per_language_report[lang]["unk_rate"] for lang in LANG_ORDER},
    "8_score": {lang: per_language_report[lang]["score"] for lang in LANG_ORDER},
    "9_target_average": {
        "languages": TARGET_LANGUAGES,
        "average_score": round(target_avg_score, 6),
        "average_fertility": round(target_avg_fertility, 6),
        "average_unk_rate": round(target_avg_unk_rate, 8),
    },
    "10_english": per_language_report["en"],
    "11_french": per_language_report["fr"],
    "12_guardrails": {
        "rule": "budget = 1.15 x mean(raw fertility of ha, sw, yo, am);"
                " breach if fertility[en] or fertility[fr] > budget (official constants)",
        "ratio": CONTEXT_FERTILITY_RATIO,
        "raw_scored_mean_fertility": round(raw_scored, 6),
        "budget": round(guardrail_budget, 6),
        "english": {"fertility": round(per_language_report["en"]["fertility"], 6),
                    "verdict": "FAIL" if "en" in guardrail_breaches else "PASS"},
        "french": {"fertility": round(per_language_report["fr"]["fertility"], 6),
                   "verdict": "FAIL" if "fr" in guardrail_breaches else "PASS"},
        "breaches": guardrail_breaches,
    },
    "14_unk_analysis": {
        "rows": unk_analysis_df.to_dict(orient="records"),
        "note": "A [UNK] appears when a character is absent from the BPE vocabulary "
                "(base alphabet = characters seen >= min_frequency=2 times in the whole train).",
    },
    "15_environment": {
        "datasets": datasets.__version__,
        "tokenizers": tokenizers.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "python": __import__("sys").version.split()[0],
    },
}

# --- Exemples de tokenisation (10 par langue) pour le rapport ---------------
_demo_rows = []
for lang in LANG_ORDER:
    for idx in select_demo_indices(val_texts[lang], RNG, k=10):
        text = val_texts[lang][idx]
        enc = tokenizer.encode(text, add_special_tokens=False)
        _demo_rows.append({
            "language": lang,
            "original": text,
            "tokens": enc.tokens,
            "token_ids": enc.ids,
        })
report["13_tokenization_examples"] = _demo_rows

MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
with open(REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print("Écrit :", REPORT_JSON)

# ---------------------------------------------------------------------------
# Rapport Markdown
# ---------------------------------------------------------------------------
def fmt_score(x):
    return f"{x:.4f}"

def fmt_unk(x):
    return f"{x:.6f}"

md_lines = []
md_lines.append("# Baseline BPE 10K — AIMS Africa Multilingual Tokenizer Challenge")
md_lines.append("")
md_lines.append(f"*Généré le {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} par "
                f"`notebooks/01_baseline_bpe_10k.ipynb`.*")
md_lines.append("")
md_lines.append("## 1. Dataset utilisé")
md_lines.append("")
md_lines.append(f"- Nom : `{DATASET_NAME}`")
md_lines.append("- URL : https://huggingface.co/datasets/Similoluwa/african-multilingual-tokenizer-challenge")
md_lines.append(f"- Révision : `{DATASET_REVISION}`")
md_lines.append(f"- Colonnes : {dataset['train'].column_names}")
md_lines.append(f"- Train : {len(dataset['train']):,} exemples — Validation : {len(dataset['validation']):,} exemples")
md_lines.append("")
md_lines.append("### Nombre d'exemples par langue")
md_lines.append("")
md_lines.append("| Langue | Code | Train | Validation |")
md_lines.append("|---|---|---:|---:|")
for lang in LANG_ORDER:
    tr = Counter(dataset["train"]["language"])[lang]
    va = Counter(dataset["validation"]["language"])[lang]
    md_lines.append(f"| {LANGUAGE_NAMES[lang]} | {lang} | {tr:,} | {va:,} |")
md_lines.append("")
md_lines.append("## 2. Statistiques du dataset")
md_lines.append("")
md_lines.append("| language | split | examples | words | characters | avg_words | avg_chars | distinct_unicode_chars |")
md_lines.append("|---|---|---:|---:|---:|---:|---:|---:|")
for row in stats_df.to_dict(orient="records"):
    md_lines.append(
        f"| {row['language']} | {row['split']} | {row['examples']:,} | {row['words']:,} | "
        f"{row['characters']:,} | {row['avg_words']:.2f} | {row['avg_chars']:.2f} | {row['distinct_unicode_chars']:,} |")
md_lines.append("")
md_lines.append("## 3. Configuration du tokenizer")
md_lines.append("")
md_lines.append("```python")
md_lines.append("from tokenizers import Tokenizer")
md_lines.append("from tokenizers.models import BPE")
md_lines.append("from tokenizers.normalizers import NFC")
md_lines.append("from tokenizers.pre_tokenizers import Whitespace")
md_lines.append("from tokenizers.trainers import BpeTrainer")
md_lines.append("")
md_lines.append('tokenizer = Tokenizer(BPE(unk_token="[UNK]"))')
md_lines.append("tokenizer.normalizer = NFC()")
md_lines.append("tokenizer.pre_tokenizer = Whitespace()")
md_lines.append("")
md_lines.append("trainer = BpeTrainer(vocab_size=10_000, min_frequency=2, special_tokens=['[UNK]'])")
md_lines.append("```")
md_lines.append("")
md_lines.append(f"- Entraînement : split `{TRAIN_SPLIT}` uniquement (multilingue).")
md_lines.append(f"- Évaluation : split `{EVAL_SPLIT}` uniquement.")
md_lines.append(f"- Définition du mot : suite maximale de caractères non-espaces (`text.split()`), identique pour les 6 langues.")
md_lines.append("- Aucune conversion ASCII / suppression d'accents / lowercase.")
md_lines.append("")
md_lines.append("## 4. Vocabulaire")
md_lines.append("")
md_lines.append(f"- Vocabulaire demandé : {VOCAB_SIZE}")
md_lines.append(f"- Vocabulaire réel : {actual_vocab_size} ({'<= 10 000, OK' if actual_vocab_size <= VOCAB_SIZE else 'DEPASSE LA LIMITE'})")
md_lines.append(f"- Tokens spéciaux : {special_in_vocab}")
md_lines.append(f"- Tokens longueur 1 / 2 / 3 : {vocab_stats['len_1']:,} / {vocab_stats['len_2']:,} / {vocab_stats['len_3']:,}")
md_lines.append(f"- Longueur moyenne / maximale : {vocab_stats['mean_length']} / {vocab_stats['max_length']}")
md_lines.append("")
md_lines.append("## 5. Résultats par langue (validation)")
md_lines.append("")
md_lines.append("| Language | Words | Tokens | Fertility | UNK | UNK Rate | Score |")
md_lines.append("|---|---:|---:|---:|---:|---:|---:|")
for lang in LANG_ORDER:
    r = per_language_report[lang]
    md_lines.append(f"| {LANGUAGE_NAMES[lang]} | {r['words']:,} | {r['tokens']:,} | "
                    f"{r['fertility']:.4f} | {r['unk']} | {r['unk_rate']:.6f} | {r['score']:.4f} |")
md_lines.append("")
md_lines.append("## 6. Target average (ha, sw, yo, am)")
md_lines.append("")
md_lines.append("| Langue | Score |")
md_lines.append("|---|---:|")
for lang in TARGET_LANGUAGES:
    md_lines.append(f"| {LANGUAGE_NAMES[lang]} | {per_language_report[lang]['score']:.4f} |")
md_lines.append(f"| **Target average** | **{target_avg_score:.4f}** |")
md_lines.append("")
md_lines.append("## 7. English / French et guardrails")
md_lines.append("")
for code, r in [("en", en), ("fr", fr)]:
    md_lines.append(f"- {LANGUAGE_NAMES[code]} : score = {r['score']:.4f} "
                    f"(fertility = {r['fertility']:.4f}, unk_rate = {r['unk_rate']:.6f}, unk = {r['unk']})")
md_lines.append("")
md_lines.append(f"- Budget guardrail = {raw_scored:.6f} x {CONTEXT_FERTILITY_RATIO} = {guardrail_budget:.6f}")
md_lines.append(f"- English guardrail : {'FAIL' if 'en' in guardrail_breaches else 'PASS'} "
                f"(fertility {en['fertility']:.6f})")
md_lines.append(f"- French guardrail : {'FAIL' if 'fr' in guardrail_breaches else 'PASS'} "
                f"(fertility {fr['fertility']:.6f})")
md_lines.append("")
md_lines.append("**Guardrail officiel :** `budget = 1.15 x moyenne(fertility brute des langues "
                "notées)` ; échec si `fertility(en)` ou `fertility(fr)` dépasse ce budget "
                "(constantes officielles `CONTEXT_FERTILITY_RATIO = 1.15`, `SCORED_LANGUAGES`, "
                "`CONTEXT_LANGUAGES`).")
md_lines.append("")
md_lines.append("## 8. Exemples de tokenisation (10 par langue)")
md_lines.append("")
for demo in _demo_rows:
    md_lines.append(f"### {LANGUAGE_NAMES[demo['language']]}")
    md_lines.append("")
    md_lines.append(f"Original : {demo['original']}")
    md_lines.append("")
    md_lines.append(f"Tokens : `{demo['tokens']}`")
    md_lines.append("")
    md_lines.append(f"Token IDs : `{demo['token_ids']}`")
    md_lines.append("")
md_lines.append("## 9. Analyse des UNK")
md_lines.append("")
md_lines.append("| language | text | problematic_token | unk_count |")
md_lines.append("|---|---|---|---:|")
if len(unk_analysis_df):
    for row in unk_analysis_df.to_dict(orient="records"):
        text_short = row["text"] if len(row["text"]) <= 120 else row["text"][:120] + "…"
        md_lines.append(f"| {row['language']} | {text_short} | {row['problematic_token']} | {row['unk_count']} |")
else:
    md_lines.append("| _(aucun [UNK] sur le split validation)_ |")
md_lines.append("")
md_lines.append(report["14_unk_analysis"]["note"])
md_lines.append("")
md_lines.append("## 10. Environnement")
md_lines.append("")
md_lines.append("| Package | Version |")
md_lines.append("|---|---|")
for k, v in report["15_environment"].items():
    md_lines.append(f"| {k} | {v} |")
md_lines.append("")

with open(REPORT_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))
print("Écrit :", REPORT_MD)

## 17. Fin — récapitulatif

Cette baseline est volontairement **non optimisée** (pas de ByteLevel, UnicodeScripts,
ParityBpeTrainer, WordPiece, Unigram, lowercase, StripAccents, NFD/NFKD, ni d'autre
`vocab_size` / `min_frequency`). Les prochaines expériences seront décidées uniquement
à partir des scores obtenus ci-dessus.

In [ ]:
# =============================================================================
# 17. Récapitulatif final
# =============================================================================
print("Pipeline terminé.")
print()
print("Artefacts produits :")
print(f"  - Tokenizer : {MODEL_FILE}  (taille vocabulaire = {actual_vocab_size})")
print(f"  - Rapport JSON  : {REPORT_JSON}")
print(f"  - Rapport MD    : {REPORT_MD}")
print()
print("Scores (validation) :")
print(results_table.to_string(index=False))
print()
print("Target average (ha, sw, yo, am) :", round(target_avg_score, 4))
print("English :", round(en["score"], 4), "| French :", round(fr["score"], 4))
print("Guardrail officiel :", "FAIL" if guardrail_breaches else "PASS",
      f"(budget {guardrail_budget:.4f} | en {en['fertility']:.4f} | fr {fr['fertility']:.4f})")

## 18. Publier les artefacts sur GitHub (script + token)

⚠️ Colab n'écrit **jamais** tout seul dans votre dépôt GitHub : il faut cloner le dépôt, y copier
les artefacts, puis committer et pousser — c'est exactement ce que fait le script ci-dessous.

**Créer le token (une seule fois) :** GitHub → *Settings* → *Developer settings* →
*Personal access tokens* → *Tokens (classic)* → **Generate new token (classic)** → portée
**`repo`** → générer et copier le token (`ghp_...`).

Deux cellules suivent :

1. **écriture du script** `push_artifacts_to_github.py` dans le répertoire courant
   (contenu identique à `scripts/push_artifacts_to_github.py` du dépôt) ;
2. **exécution du script** dans le processus du notebook → un **champ masqué**
   « Colle ton token GitHub puis valide » s'affiche. Si un secret Colab `GITHUB_TOKEN` existe
   (icône 🔑 + *Notebook access*), il est utilisé sans rien demander.

Le script est **exécuté dans le processus** (et non via `!python`) justement pour que le champ
masqué et les Secrets Colab soient accessibles. Il publie `models/**` et `reports/**` sur la
branche `arena/01a0889d-tokenizer` — **`main` reste intacte**.

Garde-fous inclus : le script **refuse de publier** des artefacts non conformes au dataset
officiel (ex. run sur données synthétiques) et **refuse** une branche inexistante. Le token est
masqué dans toutes les sorties (`***`). S'il fuite : *Settings → Developer settings → Tokens → Delete*.

### Cellule « script de publication »

La cellule suivante **écrit le script** `push_artifacts_to_github.py` dans le répertoire courant
(contenu identique à `scripts/push_artifacts_to_github.py` du dépôt), et celle d'après **l'exécute
dans le processus du notebook** — indispensable pour que le **champ masqué Colab** fonctionne et
pour que le script voie les Secrets Colab.

Le token est demandé par saisie masquée (ou lu dans le secret Colab `GITHUB_TOKEN` s'il existe).
Il n'est jamais affiché, jamais écrit sur disque, jamais commité.

Ce que le script publie : `models/**`, `reports/**` (+ `submissions/**` avec `--include-submissions`),
sur la branche `arena/01a0889d-tokenizer` (`main` reste intacte).

In [ ]:
%%writefile push_artifacts_to_github.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Publie les artefacts du challenge vers votre dépôt GitHub (méthode 2 : token).

Le token est fourni par saisie **masquée** (recommandé), par variable
d'environnement, ou par le secret Colab ``GITHUB_TOKEN``. Il n'est **jamais**
affiché, jamais écrit sur disque, jamais commité : toutes les sorties passent par
``redact()``.

Ce qui est publié (par défaut) :
    models/**      tokenizer(s) entraîné(s)
    reports/**     rapports JSON / Markdown
    submissions/** (avec --include-submissions) dossier de soumission

Exemples
--------
Colab — recommandé (dans une cellule Python, champ masqué actif) :
    import sys, runpy
    sys.argv = ["push_artifacts_to_github.py", "--source", "/content", "--include-submissions"]
    try:
        runpy.run_path("/content/push_artifacts_to_github.py", run_name="__main__")
    except SystemExit as exc:
        print("code de sortie :", exc.code)

Colab — avec !python : un sous-processus n'a ni champ masqué ni Secrets, il faut
fournir le token autrement (secret exporté dans l'environnement, ou --token-file) :
    !python scripts/push_artifacts_to_github.py --source /content

Colab, en incluant le dossier de soumission :
    !python scripts/push_artifacts_to_github.py --source /content --include-submissions

Local :
    python scripts/push_artifacts_to_github.py --repo . --source .

Vérifier sans rien publier :
    python scripts/push_artifacts_to_github.py --source . --no-push

Créer explicitement une branche inexistante :
    python scripts/push_artifacts_to_github.py --source . --branch nouvelle-branche --create-branch

Publier sur une autre branche / un autre dépôt :
    python scripts/push_artifacts_to_github.py --source . --branch main \
        --repo-url https://github.com/<user>/<repo>.git
"""

from __future__ import annotations

import argparse
import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/maick-code/tokenizer.git"
DEFAULT_BRANCH = "arena/01a0889d-tokenizer"   # branche de travail (main reste intacte)
DEFAULT_MESSAGE = "Artifacts: tokenizer.json + reports (run Colab)"
ARTIFACT_DIRS = ("models", "reports")
EXCLUDE_DIR_NAMES = {"__pycache__", ".ipynb_checkpoints", ".git"}
EXCLUDE_SUFFIXES = (".pyc", ".pyo", ".zip", ".tmp", ".log")

EXIT_OK, EXIT_ERROR, EXIT_MISSING, EXIT_UNSAFE = 0, 1, 2, 3


# --------------------------------------------------------------------------- #
# Utilitaires
# --------------------------------------------------------------------------- #
def log(message: str = "") -> None:
    print(message, flush=True)


def die(message: str, code: int) -> "NoReturn":  # noqa: F821
    log(f"\nERREUR : {message}")
    raise SystemExit(code)


def redact(text: str, token: str | None) -> str:
    """Supprime toute trace du token d'une sortie."""
    if not text:
        return ""
    if token:
        text = text.replace(token, "***")
    return text


def clone_dir_default() -> Path:
    if os.path.isdir("/content"):          # Google Colab
        return Path("/content/tokenizer")
    return Path.cwd() / ".push_clone"


# --------------------------------------------------------------------------- #
# Token
# --------------------------------------------------------------------------- #
def token_from_colab_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get("GITHUB_TOKEN")
        return value.strip() if value else None
    except Exception:
        return None


_COLAB_MASKED_FIELD_JS = r"""
new Promise((resolve) => {
  const box = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;margin-top:6px;'
                    + 'border:1px solid #c8c8c8;border-radius:6px;display:inline-block';
  const label = document.createElement('span');
  label.textContent = 'Colle ton token GitHub puis valide : ';
  const input = document.createElement('input');
  input.type = 'password';
  input.style.cssText = 'font-size:14px;padding:3px 5px;width:330px';
  const button = document.createElement('button');
  button.textContent = 'Enregistrer';
  button.style.cssText = 'margin-left:8px;padding:3px 12px';
  const done = () => {
    input.disabled = true; button.disabled = true;
    const value = input.value; box.remove(); resolve(value);
  };
  button.addEventListener('click', done);
  input.addEventListener('keydown', (event) => { if (event.key === 'Enter') done(); });
  box.appendChild(label); box.appendChild(input); box.appendChild(button);
  document.body.appendChild(box);
  input.focus();
})
"""


def token_from_colab_masked_field() -> str | None:
    """Champ de saisie masqué natif Colab (nécessite d'exécuter le script EN PROCESSUS).

    Fonctionne quand le script est lancé dans une cellule Python (``runpy``), pas
    avec ``!python`` : un sous-processus n'a pas accès à l'interface du notebook.
    """
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        value = output.eval_js(_COLAB_MASKED_FIELD_JS)
    except Exception as exc:
        log(f"Champ masqué Colab indisponible ({type(exc).__name__}) : repli sur la saisie classique.")
        return None
    if isinstance(value, str) and value.strip():
        log("Token saisi dans le champ masqué Colab (non affiché).")
        return value.strip().strip('"').strip("'")
    return None


def read_token(args: argparse.Namespace) -> str | None:
    """Token par ordre de priorité : --token-file, env, secret Colab, champ masqué Colab, saisie."""
    if args.token_file:
        path = Path(args.token_file)
        if not path.is_file():
            die(f"fichier de token introuvable : {path}", EXIT_ERROR)
        token = path.read_text(encoding="utf-8").strip()
        if token:
            log("Token lu depuis le fichier indiqué (--token-file).")
            return token

    for var in ("GITHUB_TOKEN", "GH_TOKEN"):
        token = os.environ.get(var)
        if token:
            log(f"Token récupéré depuis la variable d'environnement {var}.")
            return token.strip()

    token = token_from_colab_secret()
    if token:
        log("Token récupéré depuis le secret Colab 'GITHUB_TOKEN'.")
        return token

    if args.no_input:
        return None

    token = token_from_colab_masked_field()
    if token:
        return token

    prompt = "Colle ton token GitHub puis Entrée : "
    try:
        token = getpass.getpass(prompt)          # saisie masquée
    except Exception:
        try:
            token = input(prompt)                # repli si getpass indisponible
        except Exception:
            return None
    token = (token or "").strip().strip('"').strip("'")
    if token:
        log(f"Token saisi ({len(token)} caractères, non affiché).")
    return token or None


def authed_url(url: str, token: str | None) -> str:
    """URL https porteuse du token, uniquement pour github.com."""
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@")
    return url


# --------------------------------------------------------------------------- #
# Git
# --------------------------------------------------------------------------- #
def git(repo: Path | str | None, *args: str) -> subprocess.CompletedProcess:
    command = ["git"]
    if repo is not None:
        command += ["-C", str(repo)]
    return subprocess.run(command + list(args), capture_output=True, text=True)


def git_or_die(repo: Path | str | None, token: str | None, *args: str,
               what: str = "commande git") -> subprocess.CompletedProcess:
    result = git(repo, *args)
    if result.returncode != 0:
        die(f"{what} a échoué :\n{redact(result.stderr or result.stdout, token).strip()}",
            EXIT_ERROR)
    return result


# --------------------------------------------------------------------------- #
# Artefacts
# --------------------------------------------------------------------------- #
def collect_artifacts(source: Path, include_submissions: bool) -> list[str]:
    """Chemins relatifs (posix) des fichiers à publier, triés."""
    roots = list(ARTIFACT_DIRS) + (["submissions"] if include_submissions else [])
    files: list[str] = []
    for root in roots:
        base = source / root
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file():
                continue
            parts = set(path.relative_to(source).parts)
            if parts & EXCLUDE_DIR_NAMES or path.name.startswith("."):
                continue
            if path.suffix.lower() in EXCLUDE_SUFFIXES:
                continue
            files.append(path.relative_to(source).as_posix())
    return files


def safety_checks(source: Path, files: list[str], force: bool) -> list[str]:
    """Contrôles avant publication. Retourne la liste des avertissements bloquants."""
    import json

    problems: list[str] = []

    baseline = source / "reports" / "baseline_bpe_10k.json"
    if baseline.is_file():
        try:
            status = json.loads(baseline.read_text(encoding="utf-8")).get("status")
            if status != "computed_on_official_dataset":
                problems.append(
                    f"reports/baseline_bpe_10k.json : status = {status!r} "
                    "(run non conforme au dataset officiel)")
        except Exception as exc:
            problems.append(f"reports/baseline_bpe_10k.json illisible : {exc}")

    sweep = source / "reports" / "optimization_sweep.json"
    if sweep.is_file():
        try:
            payload = json.loads(sweep.read_text(encoding="utf-8"))
            rows = (payload.get("dataset") or {}).get("validation_rows")
            if rows != 24_000:
                problems.append(
                    f"reports/optimization_sweep.json : validation_rows = {rows} "
                    "(attendu 24 000 : le balayage n'a pas tourné sur le vrai dataset)")
        except Exception as exc:
            problems.append(f"reports/optimization_sweep.json illisible : {exc}")

    if not any(f.startswith("models/") and f.endswith("tokenizer.json") for f in files):
        problems.append("aucun models/**/tokenizer.json trouvé dans les artefacts")

    # Une soumission = UN dossier. Un dossier obsolète laissé par une exécution antérieure
    # (ancien slug, par exemple après avoir renommé SLUG) rendrait la PR invalide : le
    # checker officiel exige exactement un répertoire `submissions/<slug>/`.
    subs = source / "submissions"
    if subs.is_dir():
        slugs = sorted(p.name for p in subs.iterdir()
                       if p.is_dir() and (p / "tokenizer.json").is_file())
        if len(slugs) > 1:
            problems.append(
                f"plusieurs dossiers de soumission dans submissions/ : {', '.join(slugs)} "
                "(un seul slug est autorisé par PR ; supprimez les dossiers obsolètes)")

    if problems and not force:
        log("\n" + "!" * 74)
        log("PUBLICATION REFUSÉE — les artefacts semblent ne pas venir d'un run réel :")
        for problem in problems:
            log(f"  - {problem}")
        log("Corrigez le run, ou relancez avec --force pour publier quand même.")
        log("!" * 74)
        raise SystemExit(EXIT_UNSAFE)

    if problems:
        log("\nAVERTISSEMENT (--force) :")
        for problem in problems:
            log(f"  - {problem}")
    return problems


# --------------------------------------------------------------------------- #
# Programme principal
# --------------------------------------------------------------------------- #
def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Publie models/ et reports/ vers votre dépôt GitHub (méthode token).",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--source", default="/content" if os.path.isdir("/content") else ".",
                        help="répertoire contenant models/ et reports/ (défaut : /content ou .)")
    parser.add_argument("--repo", default=None,
                        help="clone git existant du dépôt (sinon clonage automatique)")
    parser.add_argument("--repo-url", default=DEFAULT_REPO_URL, help="URL https du dépôt")
    parser.add_argument("--branch", default=DEFAULT_BRANCH, help="branche cible")
    parser.add_argument("--message", default=DEFAULT_MESSAGE, help="message de commit")
    parser.add_argument("--token-file", default=None,
                        help="lire le token depuis un fichier (évite la saisie)")
    parser.add_argument("--no-input", action="store_true",
                        help="ne jamais demander le token de façon interactive")
    parser.add_argument("--no-push", action="store_true",
                        help="copier et commiter sans pousser")
    parser.add_argument("--include-submissions", action="store_true",
                        help="publier aussi submissions/**")
    parser.add_argument("--zip", action="store_true",
                        help="créer en plus une archive de secours dans --source")
    parser.add_argument("--force", action="store_true",
                        help="publier malgré les avertissements de conformité")
    parser.add_argument("--create-branch", action="store_true",
                        help="autoriser la création de la branche si elle n'existe pas")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    source = Path(args.source).resolve()
    branch = args.branch
    repo_url = args.repo_url

    log("=" * 74)
    log("Publication des artefacts vers GitHub")
    log("=" * 74)
    log(f"Source      : {source}")
    log(f"Dépôt       : {repo_url}")
    log(f"Branche     : {branch}")
    log(f"Artefacts   : {', '.join(ARTIFACT_DIRS + (('submissions',) if args.include_submissions else ()))}")
    log()

    files = collect_artifacts(source, args.include_submissions)
    if not files:
        die(f"aucun artefact trouvé dans {source} (attendu : models/, reports/)", EXIT_MISSING)

    log(f"{len(files)} fichier(s) à publier :")
    total = 0
    for rel in files:
        size = (source / rel).stat().st_size
        total += size
        log(f"  {size:>12,} o  {rel}")
    log(f"  {'-' * 12}")
    log(f"  {total:>12,} o  total")

    safety_checks(source, files, args.force)

    if args.zip:
        archive = source / "artifacts_backup.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            for rel in files:
                handle.write(source / rel, rel)
        log(f"\nArchive de secours : {archive} ({archive.stat().st_size:,} o)")

    token = read_token(args)

    repo = Path(args.repo).resolve() if args.repo else clone_dir_default()

    if not (repo / ".git").exists():
        if not token:
            die("aucun token fourni et pas de clone local : impossible de cloner.", EXIT_ERROR)
        log(f"\nClone de {repo_url} (branche {branch}) dans {repo} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", branch, authed_url(repo_url, token), str(repo)],
            capture_output=True, text=True)
        if clone.returncode != 0:
            die("clonage impossible (token invalide, branche inexistante ou réseau) :\n"
                f"{redact(clone.stderr or clone.stdout, token).strip()}", EXIT_ERROR)
        log("clone : OK")
    else:
        log(f"\nClone existant réutilisé : {repo}")

    if token:
        check = git(repo, "ls-remote", "--heads", authed_url(repo_url, token), branch)
        if check.returncode != 0:
            die("authentification refusée : vérifiez la portée `repo` du token,"
                " sa date d'expiration et le nom de la branche.", EXIT_ERROR)
        log("authentification : OK")

    # La branche cible doit exister : sans ce contrôle, une faute de frappe
    # créerait silencieusement une nouvelle branche distante.
    exists = git(None, "ls-remote", "--heads",
                 authed_url(repo_url, token) if token else repo_url, branch)
    if exists.returncode == 0 and not exists.stdout.strip():
        if args.create_branch:
            log(f"branche '{branch}' absente du dépôt : elle sera créée (--create-branch).")
        else:
            die(f"la branche '{branch}' n'existe pas sur {repo_url}.\n"
                "Vérifiez le nom (--branch), ou utilisez --create-branch pour la créer.",
                EXIT_ERROR)
    elif exists.returncode != 0 and not token:
        log("(impossible de vérifier la branche sans token : le push tranchera.)")

    # --- resynchronisation ---------------------------------------------------
    # Un clone Colab réutilisé (ou un clone créé dans une session précédente) peut être
    # en retard sur la branche distante : le commit local ne serait alors pas un
    # fast-forward et le push serait refusé. On se replace d'abord sur la tête distante ;
    # les artefacts étant recopiés juste après, rien n'est perdu.
    fetch = git(repo, "fetch", authed_url(repo_url, token) if token else repo_url, branch)
    if fetch.returncode == 0:
        ancestor = git(repo, "merge-base", "--is-ancestor", "FETCH_HEAD", "HEAD")
        if ancestor.returncode == 0:
            log("clone à jour avec la branche distante.")
        else:
            local = git(repo, "rev-parse", "--short", "HEAD").stdout.strip()
            remote = git(repo, "rev-parse", "--short", "FETCH_HEAD").stdout.strip()
            log(f"clone en retard ({local}) sur la branche distante ({remote}) : "
                "resynchronisation sur la tête distante (les artefacts sont recopiés ensuite).")
            git_or_die(repo, token, "checkout", "-B", branch, "FETCH_HEAD",
                       what=f"git checkout -B {branch} {remote}")
    else:
        log("fetch impossible (réseau ?) : on tente le push tel quel.")

    for rel in files:
        destination = repo / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, destination)
    log(f"{len(files)} fichier(s) copié(s) dans le clone.")

    git(repo, "config", "user.name", "Artifact Publisher")
    git(repo, "config", "user.email", "publisher@users.noreply.github.com")
    for root in {Path(rel).parts[0] for rel in files}:
        git_or_die(repo, token, "add", root, what=f"git add {root}")

    commit = git(repo, "commit", "-m", args.message)
    if commit.returncode == 0:
        log("commit : OK")
    elif "nothing to commit" in (commit.stdout + commit.stderr):
        log("commit : rien de nouveau (artefacts identiques)")
    else:
        die(f"commit impossible :\n{redact(commit.stderr or commit.stdout, token).strip()}",
            EXIT_ERROR)

    if args.no_push:
        log("\n--no-push : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    if not token:
        log("\nAucun token : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    push = git(repo, "push", authed_url(repo_url, token), f"HEAD:{branch}")
    if push.returncode != 0:
        die(f"push refusé :\n{redact(push.stderr or push.stdout, token).strip()}", EXIT_ERROR)

    log("push : OK")
    log()
    log(f"Publié sur {repo_url} (branche {branch}).")
    if "github.com" in repo_url:
        slug = repo_url.rstrip("/").removesuffix(".git")
        log(f"Vérifiez : {slug}/tree/{branch}")
    return EXIT_OK


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# =============================================================================
# Exécution du script de publication (champ masqué Colab actif)
# =============================================================================
import runpy
import sys
from pathlib import Path

SCRIPT_PATH = Path("push_artifacts_to_github.py").resolve()
assert SCRIPT_PATH.is_file(), "la cellule %%writefile ci-dessus doit être exécutée d'abord"

# --source : racine des artefacts (contient models/ et reports/)
sys.argv = [
    "push_artifacts_to_github.py",
    "--source", str(OUTPUT_ROOT),
    "--branch", "arena/01a0889d-tokenizer",
]

print("Exécution :", SCRIPT_PATH)
print("Arguments :", " ".join(sys.argv[1:]))
print("Un champ masqué « Colle ton token GitHub puis valide » va s'afficher.")
print()
try:
    runpy.run_path(str(SCRIPT_PATH), run_name="__main__")
except SystemExit as exc:
    print()
    print("Code de sortie du script :", exc.code,
          "| 0 = OK, 1 = erreur, 2 = artefacts manquants, 3 = publication refusée")